# Multi-Agent Systems and Agentic Patterns with Google ADK

In this tutorial we explore five agentic patterns (**Prompt Chaining, Routing, Parallelization, Orchestrator-Worker, Evaluator-Optimizer**) using the **Google Agent Development Kit (ADK)**.

ADK is model-agnostic: through its `LiteLlm` wrapper, ADK agents can run on non-Gemini models. Here, every agent is an ADK `LlmAgent` backed by an **Azure OpenAI** deployment (`gpt-5.4-mini` or `gpt-5.6-luna`).

---

## Setup & Installation

First, let's install the necessary packages and configure our environment.

In [1]:
# Install Google ADK, LiteLLM (used by ADK to reach Azure OpenAI) and python-dotenv
# Google ADK requires Python 3.11+
%pip install -q "google-adk>=2.12" "litellm>=1.84" python-dotenv

Note: you may need to restart the kernel to use updated packages.


### Configure Azure OpenAI Credentials

This notebook uses an **Azure OpenAI** deployment through its OpenAI-compatible **v1 endpoint**.
Credentials are read from a `.env` file in this notebook's folder (or any parent folder). Copy `example.env` to `.env` and fill it in. On **Google Colab**, upload `.env` with the 📁 *Files* panel, or just enter the values when prompted:

| Variable | Description |
|----------|-------------|
| `AZURE_OPENAI_API_KEY` | Your Azure OpenAI API key |
| `AZURE_OPENAI_ENDPOINT` | Base URL, e.g. `https://<resource>.openai.azure.com/` (the full `.../openai/v1/` URL also works) |
| `AZURE_OPENAI_DEPLOYMENT` | Deployment (model) name: `gpt-5.4-mini` or `gpt-5.6-luna` |

If a value is missing, you'll be prompted for it.

**Security Best Practice**: Never hardcode API keys in your code. Keep them in `.env` (which must never be committed or shared).

In [2]:
import getpass
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Enter your Azure OpenAI API key: ")
if not os.environ.get("AZURE_OPENAI_ENDPOINT"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = input("Enter your Azure OpenAI endpoint (https://<resource>.openai.azure.com/): ")

# Accept either the resource URL or the full v1 base URL
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/").removesuffix("/openai/v1") + "/openai/v1/"
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-mini")
print(f"Using Azure OpenAI deployment: {AZURE_OPENAI_DEPLOYMENT}")

Using Azure OpenAI deployment: gpt-5.4-mini


---

## Model Configuration

ADK connects to Azure OpenAI through `LiteLlm`, using the `openai/` provider prefix and the Azure OpenAI v1 endpoint.

We then build one reusable helper, `generate_response(prompt, agent_name)`, that:
1. creates an ADK `LlmAgent` backed by the Azure OpenAI model,
2. runs it for a single turn with a `Runner` and an in-memory session service, and
3. returns the agent's final text response.

Every "agent" in the patterns below calls this helper, so each one is a real ADK agent.

> **Note on sampling parameters:** GPT-5 family reasoning models (such as `gpt-5.6-luna`) only accept the default `temperature`, and they reject `max_tokens` / `stop`. To keep this notebook working with either deployment, we don't tune these per agent; each agent's behaviour is steered through its prompt instead.

In [3]:
import litellm
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Hide LiteLLM's provider banner for deployments missing from its model map (e.g. gpt-5.6-luna)
litellm.suppress_debug_info = True

# Azure OpenAI deployment, reached through LiteLLM's OpenAI-compatible provider
azure_model = LiteLlm(
    model=f"openai/{AZURE_OPENAI_DEPLOYMENT}",
    api_base=AZURE_OPENAI_ENDPOINT,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    num_retries=5,
)

APP_NAME = "agentic_patterns"
USER_ID = "student"


def generate_response(prompt, agent_name="assistant"):
    """Run a single-turn ADK LlmAgent on Azure OpenAI and return its final text."""
    agent = LlmAgent(name=agent_name, model=azure_model)
    runner = Runner(
        agent=agent,
        app_name=APP_NAME,
        session_service=InMemorySessionService(),
        auto_create_session=True,
    )
    message = types.Content(role="user", parts=[types.Part(text=prompt)])

    final_text = ""
    for event in runner.run(user_id=USER_ID, session_id="single_turn", new_message=message):
        if event.is_final_response() and event.content and event.content.parts:
            final_text = "".join(part.text or "" for part in event.content.parts)
    return final_text


print(f"✓ Model configured: {AZURE_OPENAI_DEPLOYMENT} (Azure OpenAI via ADK LiteLlm)")
print("✓ Ready to build agentic patterns!")

✓ Model configured: gpt-5.4-mini (Azure OpenAI via ADK LiteLlm)
✓ Ready to build agentic patterns!


---

# Pattern A: Prompt Chaining

## Concept

**Prompt Chaining** is a sequential pattern where the output of one agent becomes the input to the next agent. This creates a pipeline of transformations.

**Use Cases:**
- Content generation pipelines (draft → edit → format)
- Data transformation workflows (extract → transform → summarize)
- Multi-step reasoning tasks

**Workflow:**
```
User Input → Agent A (Process) → Output A → Agent B (Refine) → Final Output
```

## Implementation

In [4]:
# Pattern A: Prompt Chaining
# Agent A generates a blog outline, Agent B expands it into a full draft

def agent_a_outline_generator(topic):
    """Agent A: Generate a blog post outline."""
    prompt = f"""
    You are an expert content strategist. Create a detailed outline for a blog post about: {topic}

    Include:
    - A catchy title
    - 3-4 main sections with bullet points
    - A conclusion hook

    Keep it concise and well-structured.
    """

    return generate_response(prompt, agent_name="agent_a_outline_generator")


def agent_b_content_expander(outline):
    """Agent B: Expand the outline into a full blog post draft."""
    prompt = f"""
    You are a skilled content writer. Take this outline and expand it into a complete blog post:

    {outline}

    Write engaging paragraphs for each section. Keep the tone professional but conversational.
    Aim for 300-400 words total.
    """

    return generate_response(prompt, agent_name="agent_b_content_expander")


# Execute the chain
print("🔗 Prompt Chaining Example\n")
print("="*60)

topic = "The Future of Multi-Agent AI Systems"
print(f"📝 Topic: {topic}\n")

# Step 1: Agent A generates outline
print("[Agent A] Generating outline...")
outline = agent_a_outline_generator(topic)
print(f"\n✓ Outline:\n{outline}\n")
print("-"*60)

# Step 2: Agent B expands outline to full draft
print("\n[Agent B] Expanding to full draft...")
full_draft = agent_b_content_expander(outline)
print(f"\n✓ Full Draft:\n{full_draft}")

🔗 Prompt Chaining Example

📝 Topic: The Future of Multi-Agent AI Systems

[Agent A] Generating outline...

✓ Outline:
**Title:** *The Future of Multi-Agent AI Systems: How Collaborative AI Will Transform Work, Innovation, and Decision-Making*

## 1. What Multi-Agent AI Systems Are and Why They Matter
- Define multi-agent AI systems in simple terms
- Explain how they differ from single-agent AI models
- Highlight key advantages: specialization, collaboration, scalability, and resilience
- Mention current use cases across industries

## 2. Emerging Trends Shaping the Future
- Increased autonomy and coordination among agents
- More advanced communication protocols between AI agents
- Integration with large language models and tool use
- Growth of agent ecosystems for business, research, and operations

## 3. Real-World Applications and Business Impact
- Automating complex workflows across departments
- Enhancing customer support, analytics, and decision-making
- Supporting software develo

---

# Pattern B: Routing

## Concept

**Routing** uses a "Router Agent" to analyze incoming requests and direct them to the most appropriate specialized agent or tool.

**Use Cases:**
- Customer support chatbots (route to billing, technical, or sales agents)
- Multi-domain question answering
- Intent classification and task delegation

**Workflow:**
```
User Query → Router Agent → [Agent 1 | Agent 2 | Agent 3] → Response
```

## Implementation

In [5]:
# Pattern B: Routing
# Router agent decides which specialized agent to use

def router_agent(user_query):
    """Router: Analyze query and determine which agent to use."""
    prompt = f"""
    You are a routing agent. Analyze this user query and determine which specialized agent should handle it.

    Available agents:
    - MATH: For mathematical calculations and problem-solving
    - CODE: For programming and technical questions
    - CREATIVE: For creative writing, storytelling, or artistic tasks

    User query: {user_query}

    Respond with ONLY the agent name (MATH, CODE, or CREATIVE) and a brief reason (one sentence).
    Format: AGENT_NAME | Reason
    """

    return generate_response(prompt, agent_name="router_agent")


def math_agent(query):
    """Specialized agent for mathematical tasks."""
    prompt = f"""
    You are a mathematics expert. Solve this problem step-by-step:
    {query}

    Show your work clearly.
    """
    return generate_response(prompt, agent_name="math_agent")


def code_agent(query):
    """Specialized agent for coding tasks."""
    prompt = f"""
    You are a senior software engineer. Address this technical query:
    {query}

    Provide clear code examples if applicable.
    """
    return generate_response(prompt, agent_name="code_agent")


def creative_agent(query):
    """Specialized agent for creative tasks."""
    prompt = f"""
    You are a creative writer. Respond to this request with imagination and flair:
    {query}
    """
    return generate_response(prompt, agent_name="creative_agent")


def route_and_execute(user_query):
    """Route the query and execute with appropriate agent."""
    # Step 1: Route the query
    routing_decision = router_agent(user_query)
    print(f"🧭 Routing Decision: {routing_decision}\n")

    # Step 2: Extract agent name
    agent_name = routing_decision.split("|")[0].strip().upper()

    # Step 3: Execute with appropriate agent
    print(f"🤖 Executing with {agent_name} Agent...\n")

    if "MATH" in agent_name:
        return math_agent(user_query)
    elif "CODE" in agent_name:
        return code_agent(user_query)
    elif "CREATIVE" in agent_name:
        return creative_agent(user_query)
    else:
        return "Error: Could not determine appropriate agent."


# Test with different queries
print("🔀 Routing Example\n")
print("="*60)

test_queries = [
    "What is the derivative of x^2 + 3x + 5?",
    "Write a poem about artificial intelligence",
    "How do I implement a binary search in Python?"
]

for i, query in enumerate(test_queries, 1):
    print(f"\n📋 Query {i}: {query}\n")
    result = route_and_execute(query)
    print(f"✓ Result:\n{result}\n")
    print("-"*60)

🔀 Routing Example


📋 Query 1: What is the derivative of x^2 + 3x + 5?

🧭 Routing Decision: MATH | This is a calculus problem asking for a derivative.

🤖 Executing with MATH Agent...

✓ Result:
We differentiate each term separately using the power rule.

Given:
\[
f(x) = x^2 + 3x + 5
\]

### Step 1: Differentiate \(x^2\)
Using the power rule,
\[
\frac{d}{dx}(x^n) = nx^{n-1}
\]
so
\[
\frac{d}{dx}(x^2) = 2x
\]

### Step 2: Differentiate \(3x\)
The derivative of \(ax\) is just \(a\), so
\[
\frac{d}{dx}(3x) = 3
\]

### Step 3: Differentiate the constant \(5\)
The derivative of a constant is 0:
\[
\frac{d}{dx}(5) = 0
\]

### Final Answer
\[
\frac{d}{dx}(x^2 + 3x + 5) = 2x + 3
\]



------------------------------------------------------------

📋 Query 2: Write a poem about artificial intelligence

🧭 Routing Decision: CREATIVE | The user is asking for a poem, which is a creative writing task.

🤖 Executing with CREATIVE Agent...

✓ Result:
In silent rooms of midnight glass,  
A spark of though

---

# Pattern C: Parallelization

## Concept

**Parallelization** executes multiple agents or tasks simultaneously and aggregates their results. This improves efficiency for independent sub-tasks.

**Use Cases:**
- Gathering information from multiple sources
- Multi-perspective analysis
- Concurrent data processing

**Workflow:**
```
              ┌─→ Agent A ─┐
User Input ──→─→ Agent B ──→─→ Aggregator → Final Output
              └─→ Agent C ─┘
```

## Implementation

In [6]:
# Pattern C: Parallelization
# Run multiple agents concurrently and aggregate results

import asyncio
from concurrent.futures import ThreadPoolExecutor

def pros_analyzer(topic):
    """Agent that analyzes the benefits/pros of a topic."""
    prompt = f"""
    You are an optimistic analyst. List the top 3-4 benefits or advantages of: {topic}

    Be specific and provide concrete examples.
    """
    return generate_response(prompt, agent_name="pros_analyzer")


def cons_analyzer(topic):
    """Agent that analyzes the drawbacks/cons of a topic."""
    prompt = f"""
    You are a critical analyst. List the top 3-4 challenges or disadvantages of: {topic}

    Be specific and provide concrete examples.
    """
    return generate_response(prompt, agent_name="cons_analyzer")


def market_analyzer(topic):
    """Agent that analyzes market opportunities."""
    prompt = f"""
    You are a market analyst. Analyze the market potential and opportunities for: {topic}

    Focus on trends, target audiences, and growth potential.
    """
    return generate_response(prompt, agent_name="market_analyzer")


def aggregator_agent(pros, cons, market):
    """Aggregator that synthesizes all analyses into a comprehensive report."""
    prompt = f"""
    You are a synthesis expert. Combine these three analyses into a balanced, comprehensive report:

    PROS:
    {pros}

    CONS:
    {cons}

    MARKET ANALYSIS:
    {market}

    Create a well-structured summary with:
    1. Executive Summary
    2. Key Findings
    3. Recommendation
    """
    return generate_response(prompt, agent_name="aggregator_agent")


def run_parallel_analysis(topic):
    """Execute all analyses in parallel using ThreadPoolExecutor."""
    print(f"⚡ Running parallel analysis on: {topic}\n")

    # Execute agents in parallel
    with ThreadPoolExecutor(max_workers=3) as executor:
        # Submit all tasks simultaneously
        pros_future = executor.submit(pros_analyzer, topic)
        cons_future = executor.submit(cons_analyzer, topic)
        market_future = executor.submit(market_analyzer, topic)

        # Wait for all results
        print("[Parallel Execution] Running Pros, Cons, and Market analyzers...\n")

        pros = pros_future.result()
        print("✓ Pros analysis complete")

        cons = cons_future.result()
        print("✓ Cons analysis complete")

        market = market_future.result()
        print("✓ Market analysis complete\n")

    # Aggregate results
    print("[Aggregator] Synthesizing results...\n")
    final_report = aggregator_agent(pros, cons, market)

    return {
        "pros": pros,
        "cons": cons,
        "market": market,
        "final_report": final_report
    }


# Execute parallel analysis
print("⚡ Parallelization Example\n")
print("="*60)

analysis_topic = "Remote Work in Tech Companies"
results = run_parallel_analysis(analysis_topic)

print("="*60)
print("\n📊 COMPREHENSIVE REPORT:\n")
print(results["final_report"])

⚡ Parallelization Example

⚡ Running parallel analysis on: Remote Work in Tech Companies

[Parallel Execution] Running Pros, Cons, and Market analyzers...

✓ Pros analysis complete
✓ Cons analysis complete
✓ Market analysis complete

[Aggregator] Synthesizing results...


📊 COMPREHENSIVE REPORT:

## 1. Executive Summary

Remote work in tech companies has become a durable operating model, not just a temporary response to the pandemic. It offers clear advantages in talent access, flexibility, cost efficiency, and productivity, while also creating meaningful challenges around communication, collaboration, visibility, and culture. The overall market outlook remains strong, especially for companies and tools that help distributed teams hire globally, coordinate effectively, and maintain security and engagement. The best approach for many organizations is not purely remote or purely office-based, but a structured model that matches the company’s size, goals, and workflow complexity.

## 2. K

---

# Pattern D: Orchestrator-Worker

## Concept

The **Orchestrator-Worker** pattern uses a "Manager Agent" to break down complex tasks into steps and coordinate "Worker Agents" to execute them sequentially or iteratively.

**Use Cases:**
- Complex project planning and execution
- Multi-step research tasks
- Adaptive problem-solving with dynamic sub-task generation

**Workflow:**
```
Manager Agent → Plan Steps → Worker Agent (Step 1) → Worker Agent (Step 2) → ... → Final Result
```

## Implementation

In [7]:
# Pattern D: Orchestrator-Worker
# Manager breaks down task, Workers execute steps

import json
import re

def orchestrator_agent(complex_task):
    """Manager Agent: Break down complex task into actionable steps."""
    prompt = f"""
    You are a project manager AI. Break down this complex task into 3-4 clear, actionable steps:

    Task: {complex_task}

    Format your response as a numbered list:
    1. [First step description]
    2. [Second step description]
    3. [Third step description]
    ...

    Make each step specific and executable.
    """

    return generate_response(prompt, agent_name="orchestrator_agent")


def worker_agent(step_description, context=""):
    """Worker Agent: Execute a specific step."""
    prompt = f"""
    You are a worker agent. Execute this step:

    Step: {step_description}

    {f'Previous context: {context}' if context else ''}

    Provide a detailed execution report with your findings or output.
    """

    return generate_response(prompt, agent_name="worker_agent")


def parse_steps(plan):
    """Extract steps from the manager's plan."""
    # Extract numbered items
    steps = re.findall(r'\d+\.\s*(.+?)(?=\n\d+\.|$)', plan, re.DOTALL)
    return [step.strip() for step in steps]


def execute_orchestrator_worker(complex_task):
    """Execute the full orchestrator-worker pattern."""
    print(f"🎯 Complex Task: {complex_task}\n")
    print("="*60)

    # Step 1: Orchestrator creates plan
    print("\n[Orchestrator] Creating execution plan...\n")
    plan = orchestrator_agent(complex_task)
    print(f"📋 Execution Plan:\n{plan}\n")
    print("-"*60)

    # Step 2: Parse steps
    steps = parse_steps(plan)
    print(f"\n✓ Identified {len(steps)} steps\n")

    # Step 3: Workers execute each step sequentially
    results = []
    context = ""

    for i, step in enumerate(steps, 1):
        print(f"\n[Worker {i}] Executing step {i}: {step[:60]}...\n")
        result = worker_agent(step, context)
        results.append(result)

        # Build context for next step
        context += f"\nStep {i} output: {result[:200]}..."

        print(f"✓ Step {i} completed\n")
        print(f"Output preview: {result[:150]}...\n")
        print("-"*60)

    return {
        "plan": plan,
        "steps": steps,
        "results": results
    }


# Execute orchestrator-worker pattern
print("🎯 Orchestrator-Worker Example\n")
print("="*60)

complex_task = "Research and create a comprehensive guide on building a mobile app for a small business"
execution_results = execute_orchestrator_worker(complex_task)

print("\n" + "="*60)
print("\n✅ ALL STEPS COMPLETED")
print(f"\nTotal steps executed: {len(execution_results['steps'])}")

🎯 Orchestrator-Worker Example

🎯 Complex Task: Research and create a comprehensive guide on building a mobile app for a small business


[Orchestrator] Creating execution plan...

📋 Execution Plan:
1. **Define the business goals and app requirements**: Identify the small business’s target audience, core problem the app should solve, essential features, desired platforms (iOS, Android, or both), budget, timeline, and success metrics.

2. **Research the mobile app development process and options**: Gather information on app types, development approaches (native, cross-platform, no-code/low-code), required tools and technologies, typical costs, legal/privacy considerations, and common challenges for small businesses.

3. **Organize the findings into a step-by-step build plan**: Create a clear guide covering planning, wireframing, design, development, testing, launch, and maintenance, with practical recommendations tailored to a small business.

4. **Review and refine the guide with real-w

---

# Pattern E: Evaluator-Optimizer

## Concept

The **Evaluator-Optimizer** pattern implements a feedback loop where one agent generates a solution, another evaluates it, and the first agent improves based on the critique.

**Use Cases:**
- Iterative content improvement
- Code review and refinement
- Self-improving systems
- Quality assurance workflows

**Workflow:**
```
Generator → Solution v1 → Evaluator → Critique → Generator → Solution v2 → [Repeat]
```

## Implementation

In [8]:
# Pattern E: Evaluator-Optimizer
# Generator creates solution, Evaluator critiques, Generator improves

def generator_agent(task, critique=None, iteration=1):
    """Generator Agent: Create or improve a solution."""
    if iteration == 1:
        prompt = f"""
        You are a solution designer. Create a solution for this task:

        Task: {task}

        Provide a clear, detailed solution.
        """
    else:
        prompt = f"""
        You are a solution designer. Improve your previous solution based on this critique:

        Task: {task}

        Critique: {critique}

        Provide an improved solution addressing the feedback.
        """

    return generate_response(prompt, agent_name="generator_agent")


def evaluator_agent(task, solution):
    """Evaluator Agent: Critique the solution and provide feedback."""
    prompt = f"""
    You are a critical evaluator. Analyze this solution:

    Task: {task}

    Solution: {solution}

    Provide:
    1. What works well (2-3 points)
    2. What needs improvement (2-3 specific suggestions)
    3. Overall quality score (1-10)

    Be constructive but thorough.
    """

    return generate_response(prompt, agent_name="evaluator_agent")


def extract_score(critique):
    """Extract quality score from critique."""
    # Look for score pattern
    match = re.search(r'(\d+)\s*/\s*10|score.*?(\d+)', critique.lower())
    if match:
        return int(match.group(1) or match.group(2))
    return 5  # Default if not found


def run_evaluator_optimizer(task, max_iterations=3, target_score=8):
    """Execute the evaluator-optimizer feedback loop."""
    print(f"🔄 Task: {task}\n")
    print("="*60)

    solution = None
    critique = None
    iteration = 1

    while iteration <= max_iterations:
        print(f"\n🔁 ITERATION {iteration}")
        print("-"*60)

        # Generate solution
        print(f"\n[Generator] Creating solution (iteration {iteration})...\n")
        solution = generator_agent(task, critique, iteration)
        print(f"💡 Solution v{iteration}:\n{solution}\n")
        print("-"*60)

        # Evaluate solution
        print(f"\n[Evaluator] Critiquing solution...\n")
        critique = evaluator_agent(task, solution)
        print(f"📊 Critique:\n{critique}\n")

        # Check score
        score = extract_score(critique)
        print(f"\n⭐ Quality Score: {score}/10")

        if score >= target_score:
            print(f"\n✅ Target score reached! Stopping optimization.")
            break

        if iteration < max_iterations:
            print(f"\n🔄 Score below target ({target_score}). Continuing to next iteration...")

        iteration += 1

    print("\n" + "="*60)
    print(f"\n🏁 OPTIMIZATION COMPLETE after {iteration} iteration(s)")
    print(f"\nFinal score: {score}/10")

    return {
        "final_solution": solution,
        "final_critique": critique,
        "iterations": iteration,
        "final_score": score
    }


# Execute evaluator-optimizer pattern
print("🔄 Evaluator-Optimizer Example\n")
print("="*60)

task = "Design a user-friendly onboarding flow for a productivity app"
results = run_evaluator_optimizer(task, max_iterations=3, target_score=8)

print("\n" + "="*60)
print("\n📋 FINAL OPTIMIZED SOLUTION:\n")
print(results["final_solution"])

🔄 Evaluator-Optimizer Example

🔄 Task: Design a user-friendly onboarding flow for a productivity app


🔁 ITERATION 1
------------------------------------------------------------

[Generator] Creating solution (iteration 1)...

💡 Solution v1:
Below is a user-friendly onboarding flow design for a productivity app that helps new users understand value quickly, set up the app with minimal effort, and start using it successfully within the first session.

---

# Onboarding Flow Design for a Productivity App

## 1) Goals of the onboarding flow
The onboarding should:

- Help users understand the app’s main value in under 30 seconds
- Reduce setup friction
- Collect only essential preferences
- Guide users to their first meaningful action fast
- Build confidence and avoid overwhelming them
- Encourage habit formation and return usage

The best onboarding for a productivity app is usually short, personalized, and task-oriented.

---

## 2) Core principles
Design the flow around these principles

---

# Conclusion: Choosing the Right Pattern

## Pattern Selection Guide

| Pattern | Best For | Key Characteristics |
|---------|----------|--------------------|
| **Prompt Chaining** | Sequential transformations | Simple pipeline, each step depends on previous |
| **Routing** | Domain-specific queries | Classify and delegate to specialists |
| **Parallelization** | Independent sub-tasks | Run concurrently, aggregate results |
| **Orchestrator-Worker** | Complex multi-step tasks | Dynamic planning, coordinated execution |
| **Evaluator-Optimizer** | Iterative improvement | Feedback loops, quality refinement |

## When to Use Each Pattern

### 🔗 Prompt Chaining
- Content pipelines (draft → edit → publish)
- Data transformation (extract → clean → analyze)
- Sequential reasoning tasks

### 🔀 Routing
- Multi-domain chatbots
- Intent classification systems
- Customer support automation

### ⚡ Parallelization
- Multi-source research
- Comparative analysis
- Time-sensitive tasks requiring speed

### 🎯 Orchestrator-Worker
- Project planning and execution
- Complex research tasks
- Adaptive problem-solving

### 🔄 Evaluator-Optimizer
- Content quality improvement
- Code review workflows
- Self-improving systems

## Combining Patterns

These patterns can be combined for more sophisticated systems:

- **Routing + Parallelization**: Route to multiple specialists, run them in parallel
- **Orchestrator + Evaluator**: Plan steps, evaluate each before proceeding
- **Chaining + Optimization**: Create pipeline with quality checks at each stage

## Next Steps

1. **Experiment**: Try modifying the examples with your own tasks
2. **Combine**: Mix patterns to solve complex problems
3. **Optimize**: Refine prompts and agent instructions, or switch `AZURE_OPENAI_DEPLOYMENT` (e.g. to `gpt-5.6-luna`) for your use case
4. **Scale**: Add more agents, tools, and sophisticated logic
5. **Go native with ADK**: ADK's workflow agents (`SequentialAgent`, `ParallelAgent`, `LoopAgent`) and sub-agent delegation implement these patterns declaratively

---

## Additional Resources

- **Azure OpenAI v1 API Documentation**: https://learn.microsoft.com/azure/foundry/openai/api-version-lifecycle
- **Using other models (LiteLLM) in ADK**: https://adk.dev/agents/models/
- **ADK documentation**: https://adk.dev/
- **ADK workflow agents**: https://adk.dev/agents/workflow-agents/

---

*Happy building with Google ADK! 🚀*